In [10]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)
from xgboost import XGBClassifier
from IPython.display import display

In [11]:
CSV_PATH = "/kaggle/input/datasets/solarmainframe/ids-intrusion-csv/03-01-2018.csv" 


In [12]:
def load_and_clean_data(csv_path):
    print(f"Loading data from: {csv_path}")

    df = pd.read_csv(csv_path, low_memory=False)
    display(df.describe())
    if 'Dst Port' in df.columns:
        df = df[df['Dst Port'] != 'Dst Port']

    df.columns = df.columns.str.strip()

    if 'Label' in df.columns:
        df['Label'] = df['Label'].astype(str).str.strip()

    feature_cols = [col for col in df.columns if col != 'Label']
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df.dropna(axis=1, how='all', inplace=True)

    feature_cols = [col for col in df.columns if col != 'Label']

    print("Imputing missing values with column medians...")
    for col in feature_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    df.dropna(inplace=True)

    feature_cols = [col for col in df.columns if col != 'Label']
    nunique = df[feature_cols].nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    if constant_cols:
        print(f"Dropping {len(constant_cols)} constant columns...")
        df.drop(columns=constant_cols, inplace=True)

    feature_cols = [col for col in df.columns if col != 'Label']
    df[feature_cols] = df[feature_cols].astype(np.float32)

    print(f"\nFinal dataset shape: {df.shape}")
    print("\nOriginal class distribution:")
    print(df['Label'].value_counts())

    return df

In [13]:
if not os.path.exists(CSV_PATH):
    print(f"ERROR: File not found -> {CSV_PATH}")
else:
    df = load_and_clean_data(CSV_PATH)

Loading data from: /kaggle/input/datasets/solarmainframe/ids-intrusion-csv/03-01-2018.csv


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
count,331125,331125,331125,331125,331125,331125,331125,331125,331125,331125,...,331125,331125,331125,331125,331125,331125,331125,331125,331125,331125
unique,16150,4,32069,153888,329,409,3954,8806,1341,127,...,11,42186,29301,41166,27467,16390,30561,8000,19572,3
top,53,6,01/03/2018 10:51:58,2,1,1,0,0,0,0,...,20,0,0,0,0,0,0,0,0,Benign
freq,107474,212899,2050,13599,144881,161492,101483,127025,101483,219020,...,168176,285897,300908,285897,285897,276915,293857,276915,276915,238037


Imputing missing values with column medians...
Dropping 8 constant columns...

Final dataset shape: (331100, 71)

Original class distribution:
Label
Benign           238037
Infilteration     93063
Name: count, dtype: int64


In [14]:
print("\nConverting labels to binary...")
df['Label'] = df['Label'].apply(lambda x: 0 if str(x).strip() == 'Benign' else 1)

print("\nBinary class distribution:")
print(df['Label'].value_counts())


Converting labels to binary...

Binary class distribution:
Label
0    238037
1     93063
Name: count, dtype: int64


In [15]:
X = df.drop(columns=['Label']).values.astype(np.float32)
y = df['Label'].values.astype(np.int32)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


X shape: (331100, 70)
y shape: (331100,)


In [16]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("\nSplit shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)


Split shapes:
X_train: (198660, 70)
X_val  : (66220, 70)
X_test : (66220, 70)
y_train: (198660,)
y_val  : (66220,)
y_test : (66220,)


In [17]:
param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [6, 8, 16, 20, 24],
    'learning_rate': [0.05, 0.1]
}

In [18]:
best_macro_f1 = -1
best_model = None
best_params = None

print("\nStarting XGBoost tuning...\n")

for params in ParameterGrid(param_grid):
    print(f"Training XGBoost with params: {params}")

    model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        **params
    )

    model.fit(X_train, y_train)

    val_preds = model.predict(X_val)

    macro_f1 = f1_score(y_val, val_preds, average='macro')
    weighted_f1 = f1_score(y_val, val_preds, average='weighted')

    print(f"Validation Macro F1   : {macro_f1:.4f}")
    print(f"Validation Weighted F1: {weighted_f1:.4f}\n")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_model = model
        best_params = params


Starting XGBoost tuning...

Training XGBoost with params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200}
Validation Macro F1   : 0.6492
Validation Weighted F1: 0.7465

Training XGBoost with params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 400}
Validation Macro F1   : 0.6514
Validation Weighted F1: 0.7480

Training XGBoost with params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 600}
Validation Macro F1   : 0.6530
Validation Weighted F1: 0.7489

Training XGBoost with params: {'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 200}
Validation Macro F1   : 0.6520
Validation Weighted F1: 0.7484

Training XGBoost with params: {'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 400}
Validation Macro F1   : 0.6546
Validation Weighted F1: 0.7499

Training XGBoost with params: {'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 600}
Validation Macro F1   : 0.6570
Validation Weighted F1: 0.7512

Training XGBoost with params: {'learning_rate':

In [19]:
print("BEST XGBOOST PARAMETERS FOUND")

print(f"Best params: {best_params}")
print(f"Best validation Macro F1: {best_macro_f1:.4f}")

BEST XGBOOST PARAMETERS FOUND
Best params: {'learning_rate': 0.1, 'max_depth': 16, 'n_estimators': 200}
Best validation Macro F1: 0.6645


In [20]:
print("\nEvaluating best model on final test set...")

test_preds = best_model.predict(X_test)

macro_f1_test = f1_score(y_test, test_preds, average='macro')
weighted_f1_test = f1_score(y_test, test_preds, average='weighted')


print("             FINAL EVALUATION: XGBOOST            ")

print(f"Macro F1 Score   : {macro_f1_test:.4f}")
print(f"Weighted F1 Score: {weighted_f1_test:.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds))

print("\nDetailed Classification Report:")
print(classification_report(
    y_test,
    test_preds,
    target_names=['Benign', 'Malicious'],
    digits=4
))


Evaluating best model on final test set...
             FINAL EVALUATION: XGBOOST            
Macro F1 Score   : 0.6645
Weighted F1 Score: 0.7527

Confusion Matrix:
[[45877  1731]
 [12483  6129]]

Detailed Classification Report:
              precision    recall  f1-score   support

      Benign     0.7861    0.9636    0.8659     47608
   Malicious     0.7798    0.3293    0.4631     18612

    accuracy                         0.7854     66220
   macro avg     0.7829    0.6465    0.6645     66220
weighted avg     0.7843    0.7854    0.7527     66220



In [21]:
import joblib

joblib.dump(best_model, "best_xgboost_model.pkl")
print("Model saved as best_xgboost_model.pkl")

Model saved as best_xgboost_model.pkl


In [22]:
feature_names = df.drop(columns=['Label']).columns.tolist()
joblib.dump(feature_names, "feature_names.pkl")
print("Feature names saved as feature_names.pkl")

Feature names saved as feature_names.pkl


In [23]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

print("Evaluating on train and val sets to complete the table...")
train_preds = best_model.predict(X_train)
val_preds = best_model.predict(X_val)

train_acc = accuracy_score(y_train, train_preds)
val_acc = accuracy_score(y_val, val_preds)
test_acc = accuracy_score(y_test, test_preds)

train_f1 = f1_score(y_train, train_preds, average='macro')
val_f1 = f1_score(y_val, val_preds, average='macro')
test_f1 = macro_f1_test

total_instances = 331100
benign_count = 238037
malicious_count = 93063

benign_pct = (benign_count / total_instances) * 100
malicious_pct = (malicious_count / total_instances) * 100

summary_data = {
    "Attack Included": ["Infilteration"],
    "Train Accuracy": [f"{train_acc:.4f}"],
    "Val Accuracy": [f"{val_acc:.4f}"],
    "Test Accuracy": [f"{test_acc:.4f}"],
    "Train F1 (Macro)": [f"{train_f1:.4f}"],
    "Val F1 (Macro)": [f"{val_f1:.4f}"],
    "Test F1 (Macro)": [f"{test_f1:.4f}"],
    "Total Instances": [f"{total_instances:,}"],
    "Split (Train/Val/Test)": ["60% / 20% / 20%"],
    "Benign %": [f"{benign_pct:.2f}%"],
    "Malicious %": [f"{malicious_pct:.2f}%"]
}

summary_df = pd.DataFrame(summary_data)

display(summary_df)

Evaluating on train and val sets to complete the table...


,Attack Included,Train Accuracy,Val Accuracy,Test Accuracy,Train F1 (Macro),Val F1 (Macro),Test F1 (Macro),Total Instances,Split (Train/Val/Test),Benign %,Malicious %
0,Infilteration,0.8581,0.7855,0.7854,0.7919,0.6645,0.6645,"331,100",60% / 20% / 20%,71.89%,28.11%


In [25]:
import json
import joblib


feature_names = df.drop(columns=['Label']).columns.tolist()

print(f"Extracted {len(feature_names)} features.")


with open('model_columns.json', 'w') as f:
    json.dump(feature_names, f)
print("Saved: model_columns.json")

joblib.dump(best_model, 'xgboost_model.joblib')
print("Saved: xgboost_model.joblib")



Extracted 70 features.
Saved: model_columns.json
Saved: xgboost_model.joblib
